# Steam Games Market Analytics - Phân Tích Khám Phá Dữ Liệu (EDA)
**Tác giả:** Data Analyst Team  
**Dữ liệu:** Kho dữ liệu Steam Games Data Warehouse (`data/processed/steam_dw.db`)  
**Mục tiêu:** Khám phá quy mô thị trường, phân tích hành vi định giá, xu hướng thể loại và đánh giá người chơi.

In [ ]:
import os
import sqlite3
import pandas as pd

# Thiết lập hiển thị pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Đường dẫn tới Data Warehouse
db_path = os.path.abspath('../data/processed/steam_dw.db')
conn = sqlite3.connect(db_path)
print('Kết nối thành công tới SQLite Data Warehouse:', db_path)

## 1. Tổng Quan Thị Trường Toàn Nền Tảng (Executive Summary)

In [ ]:
query_kpi = """
SELECT 
    COUNT(f.appid) AS total_games,
    SUM(CASE WHEN g.is_free = 1 THEN 1 ELSE 0 END) AS total_free_games,
    SUM(CASE WHEN g.is_free = 0 THEN 1 ELSE 0 END) AS total_paid_games,
    ROUND(AVG(g.price_usd), 2) AS avg_game_price_usd,
    ROUND(SUM(f.estimated_revenue_avg_usd) / 1000000000.0, 2) AS total_market_est_revenue_billion_usd,
    SUM(f.positive_reviews) AS total_positive_reviews,
    SUM(f.negative_reviews) AS total_negative_reviews,
    ROUND(SUM(f.positive_reviews) * 100.0 / (SUM(f.positive_reviews) + SUM(f.negative_reviews)), 2) AS platform_satisfaction_rate_pct,
    MAX(f.peak_ccu) AS highest_concurrent_players
FROM fact_game_metrics f
JOIN dim_games g ON f.appid = g.appid;
"""
df_kpi = pd.read_sql_query(query_kpi, conn)
df_kpi

## 2. Top 10 Tựa Game Có Doanh Thu Ước Tính Cao Nhất

In [ ]:
query_top_rev = """
SELECT 
    game_title,
    primary_genre,
    price_usd,
    estimated_owners_avg,
    ROUND(estimated_revenue_avg_usd / 1000000.0, 2) AS est_revenue_million_usd,
    positive_ratio_pct,
    review_score_tier
FROM vw_steam_games_dashboard
ORDER BY estimated_revenue_avg_usd DESC
LIMIT 10;
"""
df_top_rev = pd.read_sql_query(query_top_rev, conn)
df_top_rev

## 3. Phân Tích Thị Phần Doanh Thu & Giá Bán Theo Thể Loại

In [ ]:
query_genres = """
SELECT 
    gen.genre_name,
    COUNT(DISTINCT bg.appid) AS total_games_count,
    ROUND(SUM(f.estimated_revenue_avg_usd) / 1000000.0, 2) AS total_est_revenue_million_usd,
    ROUND(AVG(f.estimated_revenue_avg_usd), 2) AS avg_revenue_per_game_usd,
    ROUND(AVG(g.price_usd), 2) AS avg_genre_price_usd,
    ROUND(AVG(f.positive_ratio_pct), 2) AS avg_satisfaction_pct
FROM bridge_game_genres bg
JOIN dim_genres gen ON bg.genre_id = gen.genre_id
JOIN fact_game_metrics f ON bg.appid = f.appid
JOIN dim_games g ON bg.appid = g.appid
GROUP BY gen.genre_id, gen.genre_name
ORDER BY total_est_revenue_million_usd DESC
LIMIT 10;
"""
df_genres = pd.read_sql_query(query_genres, conn)
df_genres

## 4. Tiềm Năng Của Game Hỗ Trợ Bản Địa Hóa Tiếng Việt

In [ ]:
query_vn = """
SELECT 
    CASE WHEN g.has_vietnamese = 1 THEN 'Có hỗ trợ Tiếng Việt' ELSE 'Không hỗ trợ Tiếng Việt' END AS vietnamese_support,
    COUNT(g.appid) AS total_games,
    ROUND(COUNT(g.appid) * 100.0 / (SELECT COUNT(*) FROM dim_games), 2) AS pct_of_catalog,
    ROUND(AVG(f.estimated_owners_avg), 0) AS avg_owners,
    ROUND(AVG(f.estimated_revenue_avg_usd), 2) AS avg_revenue_usd,
    ROUND(AVG(f.positive_ratio_pct), 2) AS avg_satisfaction_pct
FROM dim_games g
JOIN fact_game_metrics f ON g.appid = f.appid
GROUP BY g.has_vietnamese;
"""
df_vn = pd.read_sql_query(query_vn, conn)
df_vn